# CS-4063 <br> Natural Language Processing - Assignment 3

**Muhammad Moiz Khalid** <br>
**23i-2552**<br>
**BDS-6C**

### Github Link: https://github.com/moizkhalidd/i23-2552-NLP-Assignment3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, math, pickle, random, torch
import numpy as np, matplotlib.pyplot as plt
import torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from collections import Counter
#Setup
random.seed(42); np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
[os.makedirs(d, exist_ok=True) for d in ['models', 'results']]

In [ ]:
import gzip
import shutil
import os

drive_path = "/content/drive/MyDrive"

gz_files = [
    "cellphones.json.gz",
    "beauty.json.gz",
    "home.json.gz"
]

for gz_file in gz_files:
    gz_path = os.path.join(drive_path, gz_file)

    with gzip.open(gz_path, 'rb') as f_in:
        original_name = f_in.name.split("/")[-1].replace(".gz", "")

        output_path = os.path.join("/content", original_name)

        with open(output_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

    print(f"Extracted to: {output_path}")

## Dataset Loading and Preprocessing

In [ ]:
def loadCategoryData(filePath, maxSamples):
    records = []
    with open(filePath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                text = obj.get('reviewText', '').strip()
                rating = float(obj.get('overall', 0))
                if text and 1 <= rating <= 5:
                    records.append({'text': text, 'rating': rating})
            except (json.JSONDecodeError, ValueError):
                continue
            if len(records) >= maxSamples:
                break
    return records

In [ ]:
allRecords = []

# ~36 000 total — within 30k–45k range
catRecords = loadCategoryData('cellphones.json', 12000)
allRecords.extend(catRecords)
print('Loaded {:>6} records from {}'.format(len(catRecords), 'cellphones.json'))

catRecords = loadCategoryData('beauty.json', 12000)
allRecords.extend(catRecords)
print('Loaded {:>6} records from {}'.format(len(catRecords), 'beauty.json'))

catRecords = loadCategoryData('home.json', 12000)
allRecords.extend(catRecords)
print('Loaded {:>6} records from {}'.format(len(catRecords), 'home.json'))

print()
print('Total records loaded: {}'.format(len(allRecords)))

In [ ]:
#preprocessing
def cleanText(text):
    text = text.lower()
     # strip HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # keep alnum + apostrophe/hyphen
    text = re.sub(r'[^a-z0-9\s\'\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def simpleTokenize(text):
    #tokenization after cleaning
    return cleanText(text).split()


for rec in allRecords:
    rec['tokens'] = simpleTokenize(rec['text'])

print('Sample tokens (first record):', allRecords[0]['tokens'][:15])

In [ ]:
#split
random.shuffle(allRecords)

n = len(allRecords)
nTrain = int(n * 0.70)
nVal = int(n * 0.15)

trainData = allRecords[:nTrain]
valData = allRecords[nTrain : nTrain + nVal]
testData = allRecords[nTrain + nVal:]

print('Train size : {}'.format(len(trainData)))
print('Val size : {}'.format(len(valData)))
print('Test size : {}'.format(len(testData)))

In [ ]:
#Vocabulary
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
BOS_TOKEN = '<BOS>'
EOS_TOKEN = '<EOS>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

wordFreq = Counter()
for rec in trainData:
    wordFreq.update(rec['tokens'])

vocabWords = []
for w, c in wordFreq.most_common(30000):
    #min frequency threshold
    if c >= 3:
        vocabWords.append(w)

vocab = SPECIAL_TOKENS + vocabWords

word2idx = {}
for i, w in enumerate(vocab):
    word2idx[w] = i

idx2word = {}
for w, i in word2idx.items():
    idx2word[i] = w

PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]
BOS_IDX = word2idx[BOS_TOKEN]
EOS_IDX = word2idx[EOS_TOKEN]

VOCAB_SIZE = len(vocab)
print('Vocabulary size: {}'.format(VOCAB_SIZE))

In [ ]:
# label mapping
# sentiment: 1-2 = 0 (Negative), 3 = 1 (Neutral), 4-5 = 2 (Positive)
def ratingToSentiment(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

# derived feature: review length bucket
#Short  (< 50 tokens)  = 0
#Medium (50-150 tokens) = 1
#Long   (>150 tokens)  = 2
def lengthBucket(tokens):
    n = len(tokens)
    if n < 50:
        return 0
    elif n <= 150:
        return 1
    else:
        return 2

for rec in allRecords:
    rec['sentiment']   = ratingToSentiment(rec['rating'])
    rec['lengthClass'] = lengthBucket(rec['tokens'])

sentDist = Counter(r['sentiment']   for r in allRecords)
lenDist = Counter(r['lengthClass'] for r in allRecords)
print('Sentiment distribution : {}'.format(dict(sentDist)))
print('Length-class distribution: {}'.format(dict(lenDist)))

In [ ]:
#numericalization helpers
MAX_SEQ_LEN = 128

def encodeTokens(tokens, maxLen=MAX_SEQ_LEN):
    """Convert tokens to padded/truncated index tensor with leading CLS token."""
    # Reserve position 0 for CLS; content fills positions 1..maxLen-1
    ids = []

    #CLS token
    ids.append(UNK_IDX)

    # convert tokens to indices one by one
    for t in tokens[:maxLen - 1]:
        if t in word2idx:
            ids.append(word2idx[t])
        else:
            ids.append(UNK_IDX)

    # padding
    ids = ids + [PAD_IDX] * (maxLen - len(ids))
    return ids


class ReviewDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        inputIds = torch.tensor(encodeTokens(rec['tokens']), dtype=torch.long)
        sentLabel = torch.tensor(rec['sentiment'],   dtype=torch.long)
        lenLabel = torch.tensor(rec['lengthClass'], dtype=torch.long)
        return inputIds, sentLabel, lenLabel


trainDataset = ReviewDataset(trainData)
valDataset = ReviewDataset(valData)
testDataset = ReviewDataset(testData)

trainLoader = DataLoader(trainDataset, batch_size=64, shuffle=True)
valLoader = DataLoader(valDataset, batch_size=64, shuffle=False)
testLoader = DataLoader(testDataset, batch_size=64, shuffle=False)

print('DataLoaders ready.')